In [1]:
 !pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00


### Explanation of Installed Libraries

These three packages are commonly used together to load and run large, open-source AI models (such as LLMs) efficiently:

1. **`transformers`**: Developed by Hugging Face, this is the core library providing APIs and tools to easily download, train, and run state-of-the-art pre-trained models (like BERT, GPT, Llama, Whisper, etc.) for text, vision, and audio tasks.
2. **`accelerate`**: Another library by Hugging Face that simplifies running PyTorch models across different hardware configurations (CPU, single/multi-GPU, TPU). It automatically handles device placement and mixed-precision execution.
3. **`bitsandbytes`**: A library designed for efficient deep learning. It is widely used to quantize models (e.g., compressing 16-bit models down to 8-bit or 4-bit weights) so that large models can fit and run on consumer-grade GPUs with limited VRAM.

In [6]:
import torch
print("Is CUDA (GPU) available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

Is CUDA (GPU) available?: True
GPU Device Name: Tesla T4


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def show_vram(label):
    gb = torch.cuda.memory_allocated() / 1e9
    print(f"{label}: {gb:.2f} GB")



config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

FP16: 7.64 GB


In [5]:
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="cuda"
)
show_vram("FP16")
del model_fp16
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

FP16: 7.64 GB


In [9]:
bnb_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config_8bit, device_map="cuda"
)
show_vram("INT8")
del model_int8
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

INT8: 4.02 GB


In [11]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="cuda"
)
show_vram("INT4")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

INT4: 2.44 GB


In [12]:
prompt = "Explain what a check engine light means in one sentence."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model_int4.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Explain what a check engine light means in one sentence.

The check engine light illuminates when the vehicle's on-board diagnostics system detects an issue with the engine or related components.


## Instruction 2 (


### Loading the Model

Below, we configure a 4-bit quantization configuration using `BitsAndBytesConfig` and load the model. Using 4-bit quantization allows us to run this large model on a standard Colab T4 GPU with minimal memory usage.

### How `bitsandbytes` Quantization Works

To quantize a model during loading using the `transformers` library, we integrate with `bitsandbytes` via `BitsAndBytesConfig`. Here is a breakdown of the key parameters you can configure:

* **`load_in_4bit=True`**: Enables 4-bit quantization, converting the model's weight layers from FP16/BF16 to a compressed 4-bit representation.
* **`bnb_4bit_quant_type="nf4"`**: Specifies the quantization data type. Hugging Face recommends using NormalFloat 4 (`nf4`), which is theoretically optimal for normally distributed neural network weights.
* **`bnb_4bit_compute_dtype=torch.bfloat16`**: Sets the calculation data type. While the weights are stored in 4-bit on the GPU, they are temporarily dequantized to this data type (e.g., `bfloat16` or `float16`) during the forward pass computations to maintain speed and precision.
* **`bnb_4bit_use_double_quant=True`**: (Optional) Activating double quantization performs a second round of quantization on the quantization constants themselves, saving an extra ~0.4 GB of memory per 1B parameters.